# Automated Spoiler Detection Training
Fetches training data from backend, trains model, notifies backend. Fully automated.

1. Enable GPU (Runtime > Change runtime type > T4)
2. Set BACKEND_URL below
3. Run All

In [ ]:
BACKEND_URL = "https://infortunately-papaveraceous-asley.ngrok-free.dev"
POLL_INTERVAL = 60

import requests
r = requests.get(f"{BACKEND_URL}/api/ai/webhooks/books/needs-training", timeout=5)
print(f"Backend connected. Books needing training: {len(r.json())}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted.")

In [ ]:
import subprocess, sys
for pkg in ["unsloth", "transformers", "datasets", "trl", "accelerate", "bitsandbytes"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}" if torch.cuda.is_available() else "NO GPU - enable T4!")

In [ ]:
import json, time
from pathlib import Path

def fetch_training_data(book_id):
    r = requests.get(f"{BACKEND_URL}/api/ai/webhooks/books/{book_id}/training-data", timeout=30)
    return r.json()

def get_books_needing_training():
    r = requests.get(f"{BACKEND_URL}/api/ai/webhooks/books/needs-training", timeout=10)
    return r.json()

def notify_complete(book_id, model_name):
    requests.post(f"{BACKEND_URL}/api/ai/webhooks/training-complete",
        json={"bookId": book_id, "modelName": model_name, "status": "COMPLETE"}, timeout=10)

def notify_failed(book_id, model_name, reason):
    requests.post(f"{BACKEND_URL}/api/ai/webhooks/training-failed",
        json={"bookId": book_id, "modelName": model_name, "reason": reason}, timeout=10)

print("Functions loaded.")

In [ ]:
def train_model(book_id, examples):
    from unsloth import FastLanguageModel
    from trl import SFTTrainer
    from transformers import TrainingArguments
    from datasets import Dataset

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="unsloth/llama-3-8b-instruct", max_seq_length=2048, dtype=None, load_in_4bit=True)
    model = FastLanguageModel.get_peft_model(model, r=16,
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
        lora_alpha=16, lora_dropout=0, bias="none", use_gradient_checkpointing="unsloth")

    dataset = Dataset.from_list(examples)
    trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=dataset,
        dataset_text_field="messages", max_seq_length=2048,
        args=TrainingArguments(output_dir=f"./results/book-{book_id}", num_train_epochs=3,
            per_device_train_batch_size=2, gradient_accumulation_steps=4,
            learning_rate=2e-4, fp16=True, logging_steps=10,
            save_strategy="epoch", optim="adamw_8bit"))

    trainer.train()
    model.save_pretrained(f"./models/spoiler-{book_id}-lora")
    model.save_pretrained_gguf(f"./models/spoiler-{book_id}-gguf", tokenizer, quantization_method="q4_k_m")

    model_name = f"shelf-spoiler-book-{book_id}"
    gguf_file = list(Path(f"./models/spoiler-{book_id}-gguf").glob("*.gguf"))[0]
    modelfile = Path(f"./models/spoiler-{book_id}-gguf") / "Modelfile"
    modelfile.write_text(f'FROM {gguf_file}\n\nTEMPLATE """<|begin_of_text|><|start_header_id|>system<|end_header_id|>\nYou are a spoiler detection model. Classify reviews as SAFE, MINOR_SPOILER, or MAJOR_SPOILER. Respond with ONLY valid JSON: {{"level": "SAFE|MINOR_SPOILER|MAJOR_SPOILER", "score": 0.0-1.0, "reasoning": "..."}}<|eot_id|><|start_header_id|>user<|end_header_id|>\nAnalyze this review for spoilers: {{{{ .Prompt }}}}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n"""\n\nPARAMETER temperature 0.1\nPARAMETER num_predict 200')

    # Copy to Drive
    import shutil
    drive_path = Path(f"/content/drive/MyDrive/shelftotales-training/models/{book_id}")
    drive_path.mkdir(parents=True, exist_ok=True)
    shutil.copy2(gguf_file, drive_path / gguf_file.name)
    shutil.copy2(modelfile, drive_path / "Modelfile")

    return model_name

print("Training function ready.")

In [ ]:
print("Starting auto-training loop... Press STOP to stop.\n")
trained = set()

try:
    while True:
        books = get_books_needing_training()
        untrained = [b for b in books if b["bookId"] not in trained]

        if untrained:
            for book in untrained:
                bid = book["bookId"]
                print(f"\n{'='*50}")
                print(f"Training book {bid}...")
                try:
                    data = fetch_training_data(bid)
                    examples = data["examples"]
                    print(f"Got {len(examples)} examples from backend")
                    if len(examples) < 3:
                        print("Not enough examples, skipping.")
                        continue
                    model_name = train_model(bid, examples)
                    notify_complete(bid, model_name)
                    trained.add(bid)
                    print(f"DONE: {model_name}")
                except Exception as e:
                    print(f"FAILED: {e}")
                    notify_failed(bid, f"shelf-spoiler-book-{bid}", str(e))
        else:
            print(f"No new books. Checking in {POLL_INTERVAL}s...", end="\r")

        time.sleep(POLL_INTERVAL)
except KeyboardInterrupt:
    print("\nStopped.")